# Day 3: Dual Noise Decomposition — Physical vs. Behavioral

*Alpha Flow Research · HongJin HE · July 2026*

---

## The Core Insight

Standard models (Black-Scholes, GBM) assume a **single noise source**:
$$dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$$

This is structurally wrong. It conflates two **qualitatively different** types of uncertainty:

| Type | Source | Math Structure | Statistics |
|------|--------|----------------|------------|
| **Physical** $\tau_t$ | Information asymmetry, microstructure, fundamental valuation uncertainty | Continuous Itô integral $\sigma_\tau\,dW_t^{(\tau)}$ | Locally Gaussian, continuous paths |
| **Behavioral** $\eta_t$ | Panic, herding, flash crashes, sudden policy shifts | Pure-jump Lévy process $dJ_t^{(\eta)}$ | Non-Gaussian, **discontinuous** jumps |

**Definition (Dual Noise)**:
$$d\mathbf{S}_t = \mu(\mathbf{S}_t)\,dt + \underbrace{\sigma_\tau(\mathbf{S}_t)\,dW_t^{(\tau)}}_{\text{physical noise}} + \underbrace{dJ_t^{(\eta)}}_{\text{behavioral noise}}$$

The two terms are **orthogonal**: $[W^{(\tau)}, J^{(\eta)}]_t = 0$ a.s.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

np.random.seed(42)
T, dt = 1.0, 1/252   # 1 year, daily
n = int(T / dt)

# Physical noise: Brownian motion
sigma_tau = 0.20 * np.sqrt(dt)
W = np.random.randn(n) * sigma_tau

# Behavioral noise: compound Poisson (Lévy jumps)
lambda_eta = 5.0     # 5 jumps/year
jump_size  = 0.025   # avg |jump| = 2.5%
poisson_mask = np.random.poisson(lambda_eta * dt, n)
J = np.array([sum(np.random.choice([-1,1]) * np.random.exponential(jump_size)
                  for _ in range(k)) for k in poisson_mask])

# Combined process
mu = 0.08 * dt
returns = mu + W + J
prices_physical    = 100 * np.exp(np.cumsum(mu + W))
prices_behavioral  = 100 * np.exp(np.cumsum(mu + J))
prices_combined    = 100 * np.exp(np.cumsum(returns))

t = np.linspace(0, 1, n)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Price paths
ax = axes[0, 0]
ax.plot(t, prices_physical,   color='#2E86AB', alpha=0.8, linewidth=1.2, label='Physical only (Brownian)')
ax.plot(t, prices_behavioral, color='#C73E1D', alpha=0.8, linewidth=1.2, label='Behavioral only (Lévy jumps)')
ax.plot(t, prices_combined,   color='#333333', linewidth=2.0, label='Combined (Dual Noise)')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Price')
ax.set_title('Price Paths: Physical vs. Behavioral Noise', fontweight='bold')
ax.legend(fontsize=9)

# Top-right: Return distributions
ax = axes[0, 1]
bins = np.linspace(-0.08, 0.08, 80)
ax.hist(W, bins=bins, alpha=0.6, color='#2E86AB', label='Physical (Gaussian)', density=True)
ax.hist(J, bins=bins, alpha=0.6, color='#C73E1D', label='Behavioral (Lévy)', density=True)
ax.hist(returns - mu, bins=bins, alpha=0.4, color='gray', label='Combined', density=True)

# Gaussian overlay
from scipy.stats import norm
x = np.linspace(-0.08, 0.08, 200)
ax.plot(x, norm.pdf(x, 0, sigma_tau), 'b-', linewidth=2, label='N(0,σ²) fit')
ax.set_xlabel('Daily return'); ax.set_ylabel('Density')
ax.set_title('Return Distributions: Fat Tails from Behavioral Noise', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(-0.08, 0.08)

# Bottom-left: Bipower Variation estimate of σ_τ
ax = axes[1, 0]
window = 21
bpv_estimates   = []
rv_estimates    = []
true_sigma_sq   = sigma_tau**2 * np.ones(n - window)

for i in range(n - window):
    r = returns[i:i+window]
    rv  = np.sum(r**2)
    bpv = (np.pi/2) * np.sum(np.abs(r[1:]) * np.abs(r[:-1]))
    bpv_estimates.append(bpv)
    rv_estimates.append(rv)

ax.plot(t[:n-window], rv_estimates,   color='gray',    alpha=0.7, linewidth=1, label='Realized Variance (RV)')
ax.plot(t[:n-window], bpv_estimates,  color='#2E86AB', linewidth=1.5, label='Bipower Variation (BPV) — jump-robust')
ax.plot(t[:n-window], true_sigma_sq * window, color='#C73E1D', linewidth=2, linestyle='--', label='True σ_τ² (physical only)')
ax.fill_between(t[:n-window], bpv_estimates, rv_estimates, alpha=0.2, color='orange', label='Jump contribution J̃')
ax.set_xlabel('Time'); ax.set_ylabel('Variance (21-day window)')
ax.set_title('BPV Separates Physical from Behavioral Variance', fontweight='bold')
ax.legend(fontsize=8)

# Bottom-right: Q-Q plot showing fat tails from behavioral noise
ax = axes[1, 1]
from scipy.stats import probplot
probplot(returns, dist='norm', plot=ax)
ax.set_title('Q-Q Plot: Behavioral Jumps Create Heavy Tails\n(deviation from straight line = non-Gaussianity)', fontweight='bold')
ax.get_lines()[0].set(color='#333333', alpha=0.5)
ax.get_lines()[1].set(color='#C73E1D', linewidth=2)

plt.tight_layout()
plt.savefig('notebooks/fig_dual_noise.png', dpi=150, bbox_inches='tight')
plt.show()

## Calibrating the Dual Noise Parameters

From our `state/noise.py` module:

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from state.noise import DualNoiseCalibrator

calibrator = DualNoiseCalibrator(alpha_lm=0.001)
params = calibrator.calibrate(returns, dt=dt)

print("=" * 50)
print("Dual Noise Calibration Results")
print("=" * 50)
print(f"  σ_τ  (physical vol)      = {params.sigma_tau:.5f}/day  (true: {sigma_tau:.5f})")
print(f"  λ_η  (jump intensity)    = {params.lambda_eta:.5f}/day  (true: {lambda_eta*dt:.5f})")
print(f"  m₂^η (jump 2nd moment)   = {params.m2_eta:.6f}")
print(f"  τ_t  (composite temp)    = {params.tau_t:.5f}")
print(f"")
print(f"  Cramér-Rao 1-day bound   ≥ {params.sigma_tau**2 + params.lambda_eta * params.m2_eta:.6f}")
print(f"  → Even perfect model cannot predict below this floor")
print("=" * 50)

## Why This Matters for the World Model

The dual noise decomposition tells us:

1. **What is learnable**: The drift $\mu(\mathbf{S}_t)$ — the predictable component (Doob-Meyer: $A_t$)
2. **What is fundamentally unpredictable**: $M_t$ (martingale) — the noise floor
3. **How to set realistic goals**: The Cramér-Rao bound sets the minimum achievable prediction error

The composite temperature parameter:
$$\tau_t = \sqrt{\sigma_\tau^2(\mathbf{S}_t) + \lambda_\eta(t) \cdot m_2^\eta(t)}$$

is **time-varying** — high during crises (large $\lambda_\eta$), low during calm periods. The world model's honest acknowledgment of its own uncertainty.

**Next**: Day 4 — State Space and Information Sets (`day04_information_world_state_space.ipynb`)